# Step 3: Composite Index and Visual Analysis

This notebook combines all five pillar scores into a unified MGNREGA Impact Index and generates analytical and visual outputs in `final_outputs/`.

It supports two input formats:
- `region_id, year, pillar_score`
- `region_id, pillar_score` (expanded to panel years using base panel data)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_PANEL_PATH = Path('Panel_Data 2014-24.csv')
PILLAR_DIR = Path('outputs')
FINAL_DIR = Path('final_outputs')
FINAL_DIR.mkdir(exist_ok=True)

pillar_files = {
    'shock': PILLAR_DIR / 'pillar1_shock_score.csv',
    'distribution': PILLAR_DIR / 'pillar2_distribution_score.csv',
    'income': PILLAR_DIR / 'pillar3_income_score.csv',
    'distortion': PILLAR_DIR / 'pillar4_distortion_score.csv',
    'structural': PILLAR_DIR / 'pillar5_structural_score.csv',
}

weights = {
    'shock': 0.25,
    'distribution': 0.20,
    'income': 0.20,
    'distortion': 0.15,
    'structural': 0.20,
}

expected_pillars = list(weights.keys())

In [2]:
def standardize_pillar_file(path: Path, pillar_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    score_cols = [c for c in df.columns if c not in ['region_id', 'year']]
    if len(score_cols) != 1:
        raise ValueError(f'{path.name}: expected one score column, got {score_cols}')
    score_col = score_cols[0]
    df = df.rename(columns={score_col: pillar_name})
    if 'year' in df.columns:
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    return df[['region_id'] + (['year'] if 'year' in df.columns else []) + [pillar_name]]


def build_panel_scaffold(base_panel_path: Path) -> pd.DataFrame:
    base = pd.read_csv(base_panel_path)
    base['region_id'] = base['District'].astype(str).str.strip() + '_' + base['State'].astype(str).str.strip()
    base['year'] = pd.to_numeric(base['Year'], errors='coerce').astype('Int64')
    return base[['region_id', 'year']].dropna().drop_duplicates().sort_values(['region_id', 'year'])


def minmax_clip_01(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors='coerce')
    mn, mx = s.min(skipna=True), s.max(skipna=True)
    if pd.isna(mn) or pd.isna(mx) or mn == mx:
        out = pd.Series(np.full(len(s), 0.5), index=s.index)
    else:
        out = (s - mn) / (mx - mn)
    return out.clip(0, 1)


def radar_plot(values_dict, title, outpath):
    labels = list(values_dict.keys())
    values = [float(values_dict[k]) for k in labels]
    values = [min(1.0, max(0.0, v)) for v in values]
    values += values[:1]

    angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    ax.plot(angles, values, linewidth=2)
    ax.fill(angles, values, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([l.title() for l in labels])
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_ylim(0, 1)
    ax.set_title(title)
    plt.tight_layout()
    fig.savefig(outpath, dpi=300)
    plt.close(fig)

In [3]:
# Load and merge all pillar files
pillar_dfs = {k: standardize_pillar_file(v, k) for k, v in pillar_files.items()}
has_year = all('year' in df.columns for df in pillar_dfs.values())

if has_year:
    merged = None
    for k in expected_pillars:
        if merged is None:
            merged = pillar_dfs[k].copy()
        else:
            merged = merged.merge(pillar_dfs[k], on=['region_id', 'year'], how='inner')
else:
    scaffold = build_panel_scaffold(BASE_PANEL_PATH)
    merged = scaffold.copy()
    for k in expected_pillars:
        tmp = pillar_dfs[k][['region_id', k]].drop_duplicates('region_id')
        merged = merged.merge(tmp, on='region_id', how='left')

for c in expected_pillars:
    merged[c] = minmax_clip_01(merged[c])

merged = merged.dropna(subset=expected_pillars).copy()
merged['year'] = pd.to_numeric(merged['year'], errors='coerce').astype(int)
merged = merged.sort_values(['region_id', 'year']).reset_index(drop=True)
merged.head()

,region_id,year,shock,distribution,income,distortion,structural
0,ADILABAD_TELANGANA,2015,0.737443,0.461004,0.461007,0.538993,0.693617
1,ADILABAD_TELANGANA,2016,0.648405,0.446014,0.446017,0.553983,0.667772
2,ADILABAD_TELANGANA,2017,0.828035,0.430083,0.430086,0.569914,0.678805
3,ADILABAD_TELANGANA,2018,0.530512,0.429317,0.429320,0.570680,0.690530
4,ADILABAD_TELANGANA,2019,0.594376,0.404553,0.404555,0.595445,0.691093


In [4]:
# Step 2: Composite index (equal and weighted)
merged['index_equal_weight'] = merged[expected_pillars].mean(axis=1)
merged['index_weighted'] = sum(merged[k] * w for k, w in weights.items())
merged['MGNREGA_Index'] = merged['index_equal_weight']

composite_cols = ['region_id', 'year'] + expected_pillars + ['index_equal_weight', 'index_weighted', 'MGNREGA_Index']
composite_index = merged[composite_cols].copy()
composite_index.to_csv(FINAL_DIR / 'composite_index.csv', index=False)
composite_index.head()

,region_id,year,shock,distribution,income,distortion,structural,index_equal_weight,index_weighted,MGNREGA_Index
0,ADILABAD_TELANGANA,2015,0.737443,0.461004,0.461007,0.538993,0.693617,0.578413,0.588335,0.578413
1,ADILABAD_TELANGANA,2016,0.648405,0.446014,0.446017,0.553983,0.667772,0.552438,0.557159,0.552438
2,ADILABAD_TELANGANA,2017,0.828035,0.430083,0.430086,0.569914,0.678805,0.587385,0.600291,0.587385
3,ADILABAD_TELANGANA,2018,0.530512,0.429317,0.429320,0.570680,0.690530,0.530072,0.528064,0.530072
4,ADILABAD_TELANGANA,2019,0.594376,0.404553,0.404555,0.595445,0.691093,0.538005,0.537951,0.538005


In [5]:
# Step 3: Time evolution analysis
time_trend = merged.groupby('year', as_index=False).agg(
    MGNREGA_Index=('MGNREGA_Index', 'mean'),
    shock=('shock', 'mean'),
    distribution=('distribution', 'mean'),
    income=('income', 'mean'),
    distortion=('distortion', 'mean'),
    structural=('structural', 'mean'),
)
time_trend.to_csv(FINAL_DIR / 'time_trend.csv', index=False)

# Plot: year vs composite index
plt.figure(figsize=(10, 5))
plt.plot(time_trend['year'], time_trend['MGNREGA_Index'], marker='o')
plt.title('Year-wise Mean MGNREGA Composite Index')
plt.xlabel('Year')
plt.ylabel('Mean Composite Index')
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FINAL_DIR / 'time_trend_index.png', dpi=300)
plt.close()

# Plot: year vs each pillar
plt.figure(figsize=(12, 6))
for c in expected_pillars:
    plt.plot(time_trend['year'], time_trend[c], marker='o', label=c.title())
plt.title('Year-wise Mean Scores by Pillar')
plt.xlabel('Year')
plt.ylabel('Mean Pillar Score')
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(FINAL_DIR / 'time_trend_pillars.png', dpi=300)
plt.close()

time_trend.head()

,year,MGNREGA_Index,shock,distribution,income,distortion,structural
0,2015,0.556789,0.654632,0.282715,0.282716,0.717284,0.846597
1,2016,0.549765,0.648481,0.301789,0.301790,0.698210,0.798556
2,2017,0.549306,0.651833,0.312569,0.312570,0.687430,0.782127
3,2018,0.557508,0.674159,0.336390,0.336392,0.663608,0.776993
4,2019,0.525852,0.550253,0.319575,0.319577,0.680423,0.759433


In [6]:
# Step 4: Pre vs Post COVID analysis
merged['COVID_dummy'] = merged['year'].isin([2020, 2021]).astype(int)
merged['period'] = np.where(merged['year'].between(2014, 2019), 'Pre-COVID', 'Post-COVID')

covid_comparison = merged.groupby('period', as_index=False).agg(
    shock=('shock', 'mean'),
    distribution=('distribution', 'mean'),
    income=('income', 'mean'),
    distortion=('distortion', 'mean'),
    structural=('structural', 'mean'),
    MGNREGA_Index=('MGNREGA_Index', 'mean'),
)
covid_comparison.to_csv(FINAL_DIR / 'covid_comparison.csv', index=False)

# Pre vs post bar chart for pillars
plot_df = covid_comparison.set_index('period')[expected_pillars].T
plot_df.plot(kind='bar', figsize=(10, 6))
plt.title('Pre vs Post COVID: Pillar Means')
plt.xlabel('Pillar')
plt.ylabel('Mean Score')
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FINAL_DIR / 'pre_post_covid_pillars.png', dpi=300)
plt.close()

covid_comparison

,period,shock,distribution,income,distortion,structural,MGNREGA_Index
0,Post-COVID,0.593113,0.383908,0.383909,0.616091,0.727545,0.540913
1,Pre-COVID,0.635871,0.310608,0.310609,0.689391,0.792741,0.547844


In [7]:
# Step 5: Radar charts (overall, pre, post)
overall_vals = merged[expected_pillars].mean().to_dict()
pre_vals = merged.loc[merged['period'] == 'Pre-COVID', expected_pillars].mean().to_dict()
post_vals = merged.loc[merged['period'] == 'Post-COVID', expected_pillars].mean().to_dict()

radar_plot(overall_vals, 'Radar: Overall Average', FINAL_DIR / 'radar_overall.png')
radar_plot(pre_vals, 'Radar: Pre-COVID Average', FINAL_DIR / 'radar_pre_covid.png')
radar_plot(post_vals, 'Radar: Post-COVID Average', FINAL_DIR / 'radar_post_covid.png')

pd.DataFrame([overall_vals, pre_vals, post_vals], index=['overall', 'pre', 'post'])

,shock,distribution,income,distortion,structural
overall,0.614492,0.347258,0.347259,0.652741,0.760143
pre,0.635871,0.310608,0.310609,0.689391,0.792741
post,0.593113,0.383908,0.383909,0.616091,0.727545


In [8]:
# Step 6: Regional comparison (top/bottom 5)
region_rank = merged.groupby('region_id', as_index=False).agg(
    MGNREGA_Index=('MGNREGA_Index', 'mean'),
    shock=('shock', 'mean'),
    distribution=('distribution', 'mean'),
    income=('income', 'mean'),
    distortion=('distortion', 'mean'),
    structural=('structural', 'mean'),
)
region_rank = region_rank.sort_values('MGNREGA_Index', ascending=False).reset_index(drop=True)
region_rank['rank_desc'] = np.arange(1, len(region_rank) + 1)

top5 = region_rank.head(5).copy()
bottom5 = region_rank.tail(5).copy()
regional_comparison = pd.concat([
    top5.assign(group='Top 5'),
    bottom5.assign(group='Bottom 5')
], ignore_index=True)
regional_comparison.to_csv(FINAL_DIR / 'regional_comparison.csv', index=False)

best_region = top5.iloc[0]
worst_region = bottom5.iloc[-1]
radar_plot(best_region[expected_pillars].to_dict(), f'Best Region: {best_region["region_id"]}', FINAL_DIR / 'radar_best_region.png')
radar_plot(worst_region[expected_pillars].to_dict(), f'Worst Region: {worst_region["region_id"]}', FINAL_DIR / 'radar_worst_region.png')

regional_comparison

,region_id,MGNREGA_Index,shock,distribution,income,distortion,structural,rank_desc,group
0,VIZIANAGARAM_ANDHRA PRADESH,0.572737,0.612839,9.439412e-01,0.943943,0.056057,0.306906,1,Top 5
1,BALRAMPUR_CHHATTISGARH,0.572042,0.642388,6.913718e-01,0.691374,0.308626,0.526451,2,Top 5
2,SRIKAKULAM_ANDHRA PRADESH,0.571702,0.620589,8.057072e-01,0.805709,0.194291,0.432215,3,Top 5
3,VELLORE_TAMIL NADU,0.571101,0.649367,7.680344e-01,0.768038,0.231962,0.438102,4,Top 5
4,THANJAVUR_TAMIL NADU,0.568857,0.617445,7.446578e-01,0.744662,0.255338,0.482184,5,Top 5
5,BHARUCH_GUJARAT,0.527203,0.618010,4.261476e-02,0.042614,0.957386,0.975392,252,Bottom 5
6,KOLHAPUR_MAHARASHTRA,0.526521,0.620708,1.645021e-02,0.016449,0.983551,0.995448,253,Bottom 5
7,SOLAPUR_MAHARASHTRA,0.526185,0.615758,2.238334e-02,0.022382,0.977618,0.992785,254,Bottom 5
8,REWARI_HARYANA,0.526034,0.616812,2.372650e-02,0.023726,0.976274,0.989632,255,Bottom 5
9,GHAZIABAD_UTTAR PRADESH,0.523563,0.619888,7.074654e-07,0.000000,1.000000,0.997925,256,Bottom 5


In [9]:
# Step 7: Correlation analysis
corr_cols = expected_pillars + ['MGNREGA_Index']
correlation_matrix = merged[corr_cols].corr()
correlation_matrix.to_csv(FINAL_DIR / 'correlation_matrix.csv')
correlation_matrix

,shock,distribution,income,distortion,structural,MGNREGA_Index
shock,1.000000,-0.042889,-0.042889,0.042889,0.121944,0.743068
distribution,-0.042889,1.000000,1.000000,-1.000000,-0.878923,0.439502
income,-0.042889,1.000000,1.000000,-1.000000,-0.878923,0.439502
distortion,0.042889,-1.000000,-1.000000,1.000000,0.878923,-0.439502
structural,0.121944,-0.878923,-0.878923,0.878923,1.000000,-0.099176
MGNREGA_Index,0.743068,0.439502,0.439502,-0.439502,-0.099176,1.000000


In [10]:
# Step 8: Save final master dataset
final_analysis_dataset = merged[['region_id', 'year'] + expected_pillars + ['index_equal_weight', 'index_weighted', 'MGNREGA_Index', 'COVID_dummy', 'period']].copy()
final_analysis_dataset.to_csv(FINAL_DIR / 'final_analysis_dataset.csv', index=False)
final_analysis_dataset.head()

,region_id,year,shock,distribution,income,distortion,structural,index_equal_weight,index_weighted,MGNREGA_Index,COVID_dummy,period
0,ADILABAD_TELANGANA,2015,0.737443,0.461004,0.461007,0.538993,0.693617,0.578413,0.588335,0.578413,0,Pre-COVID
1,ADILABAD_TELANGANA,2016,0.648405,0.446014,0.446017,0.553983,0.667772,0.552438,0.557159,0.552438,0,Pre-COVID
2,ADILABAD_TELANGANA,2017,0.828035,0.430083,0.430086,0.569914,0.678805,0.587385,0.600291,0.587385,0,Pre-COVID
3,ADILABAD_TELANGANA,2018,0.530512,0.429317,0.429320,0.570680,0.690530,0.530072,0.528064,0.530072,0,Pre-COVID
4,ADILABAD_TELANGANA,2019,0.594376,0.404553,0.404555,0.595445,0.691093,0.538005,0.537951,0.538005,0,Pre-COVID


In [11]:
# Step 9: Summary tables
summary_stats = merged[expected_pillars + ['MGNREGA_Index']].agg(['mean', 'std']).T.reset_index().rename(columns={'index': 'metric'})
summary_stats.to_csv(FINAL_DIR / 'summary_mean_std.csv', index=False)

pillar_importance = pd.DataFrame([
    {'pillar': 'shock', 'weight': weights['shock']},
    {'pillar': 'distribution', 'weight': weights['distribution']},
    {'pillar': 'income', 'weight': weights['income']},
    {'pillar': 'distortion', 'weight': weights['distortion']},
    {'pillar': 'structural', 'weight': weights['structural']},
]).sort_values('weight', ascending=False)
pillar_importance.to_csv(FINAL_DIR / 'summary_pillar_ranking.csv', index=False)

covid_wide = covid_comparison.set_index('period')
if {'Pre-COVID', 'Post-COVID'}.issubset(set(covid_wide.index)):
    change_post_pre = (covid_wide.loc['Post-COVID', expected_pillars + ['MGNREGA_Index']] - covid_wide.loc['Pre-COVID', expected_pillars + ['MGNREGA_Index']])
    change_post_pre = change_post_pre.reset_index()
    change_post_pre.columns = ['metric', 'post_minus_pre']
else:
    change_post_pre = pd.DataFrame({'metric': expected_pillars + ['MGNREGA_Index'], 'post_minus_pre': np.nan})
change_post_pre.to_csv(FINAL_DIR / 'summary_post_minus_pre.csv', index=False)

print('Final outputs generated in:', FINAL_DIR.resolve())
summary_stats.head()

Final outputs generated in: C:\Users\Tanishq op\Downloads\MGNREGA Analysis\final_outputs


,metric,mean,std
0,shock,0.614492,0.109395
1,distribution,0.347258,0.218978
2,income,0.347259,0.218979
3,distortion,0.652741,0.218979
4,structural,0.760143,0.163122
